# AFHQ Autoencoder

A DC-AE-style convolutional autoencoder for AFHQ animal faces, trained with a **three-phase curriculum**:

1. **Phase 1** — L1 + LPIPS on 64×64, train the *full* model.
2. **Phase 2** — L1 + LPIPS on 256×256, train *only* the latent bottleneck (`to_latent`, `from_latent`) so it adapts to high resolution.
3. **Phase 3** — L1 + LPIPS + PatchGAN on 64×64, train *only* the output head (`out_from_channels`) to sharpen texture.

The three phases share one in-memory model, so weights carry over. **Validation is always done at 256×256** (the model is fully convolutional, so it runs at any resolution) and reports L1, LPIPS, PSNR, SSIM, and reconstruction-FID (rFID).

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import torch
from lightning import Trainer, seed_everything
from torchinfo import summary
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

from chimera.data import AFHQDataModule
from chimera.models import PatchGANDiscriminator, PetPaletteAE
from chimera.modules import AdversarialAutoencoderModule, AutoencoderModule
from chimera.optim import LinearWarmupCosineAnnealingLR

os.environ["DATA_DIR"] = "../../../data"
# Keep the LPIPS-VGG and FID-Inception weights off the small root disk.
os.environ.setdefault("HF_HOME", "/mnt/ai/data/hf")
os.environ.setdefault("TORCH_HOME", "/mnt/ai/data/torch")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")

# Reproducibility: seed all RNGs (incl. dataloader workers). Pair with
# Trainer(deterministic=True) below for deterministic CUDA kernels too.
SEED = 42
seed_everything(SEED, workers=True)

## Data

Two AFHQ views of the same split — 64×64 for phases 1 & 3, 256×256 for phase 2 and for **all** validation. `normalize=False` keeps pixels in `[0, 1]` to match the decoder's sigmoid output and the `data_range=1.0` metrics (LPIPS/FID take `[0, 1]` inputs too). Both datamodules use the same `seed`, so their train/val split indices line up.

In [ ]:
def make_dm(image_size, batch_size):
    return AFHQDataModule(
        data_dir=os.environ["DATA_DIR"],
        image_size=image_size,
        batch_size=batch_size,
        num_workers=8,
        normalize=False,
        seed=SEED,
    )


dm64 = make_dm(64, batch_size=64)
dm256 = make_dm(256, batch_size=32)
for dm in (dm64, dm256):
    dm.prepare_data()
    dm.setup("fit")

train_loader_64 = dm64.train_dataloader()
train_loader_256 = dm256.train_dataloader()
val_loader = dm256.val_dataloader()  # validation is always 256x256
print(f"classes={dm256.classes}")

In [ ]:
images, _ = next(iter(val_loader))

grid = make_grid(images[:8].clamp(0, 1), nrow=8)
plt.figure(figsize=(12, 2))
plt.imshow(to_pil_image(grid))
plt.title("Sample AFHQ Images (64x64)")
plt.axis("off")
plt.show()

## Model

`PetPaletteAE` is a DC-AE-style convolutional autoencoder (pixel-unshuffle downsampling / pixel-shuffle upsampling with residual shortcuts). It is fully convolutional, so it works at any resolution: three downsampling blocks give a `(latent_dim, H/8, W/8)` spatial latent, and the decoder reconstructs through a sigmoid to `[0, 1]`. The `to_latent` / `from_latent` `1×1` convs (phase 2) and the `out_from_channels` head (phase 3) are the submodules the later phases fine-tune.

In [ ]:
model = PetPaletteAE(
    in_channels=3, latent_dim=4, base_channels=32, fsq_levels=[8, 5, 5, 5]
)
summary(
    model,
    input_size=(1, 3, 64, 64),
    col_names=["output_size", "mult_adds", "num_params"],
    depth=1,
    verbose=0,
)

## Phase 1 — L1 + LPIPS @ 64×64 (full model)

Train every parameter on 64×64 reconstruction with an L1 + VGG-LPIPS loss.

In [ ]:
PHASE1_EPOCHS = 10

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = LinearWarmupCosineAnnealingLR(
    optimizer,
    warmup_steps=100,
    n_epochs=PHASE1_EPOCHS,
    train_loader=train_loader_64,
)

p1 = AutoencoderModule(model, optimizer, scheduler, lpips_weight=1.0, compute_rfid=True)

trainer = Trainer(
    max_epochs=PHASE1_EPOCHS,
    precision="bf16-mixed",
    gradient_clip_algorithm="norm",
    gradient_clip_val=1.0,
    deterministic=True,
    check_val_every_n_epoch=2,
)
trainer.fit(p1, train_dataloaders=train_loader_64, val_dataloaders=val_loader)
trainer.test(p1, dataloaders=val_loader)

## Phase 2 — L1 + LPIPS @ 256×256 (bottleneck only)

Freeze everything except the `to_latent` / `from_latent` `1×1` convs and train them at full 256×256 resolution, adapting the latent bottleneck to high-res inputs. `train_only` also holds the frozen BatchNorm running stats fixed.

In [ ]:
PHASE2_EPOCHS = 15
TRAIN_ONLY_P2 = ["to_latent", "from_latent"]

params = AutoencoderModule.trainable_params(model, TRAIN_ONLY_P2)
optimizer = torch.optim.AdamW(params, lr=1e-3)
scheduler = LinearWarmupCosineAnnealingLR(
    optimizer,
    warmup_steps=100,
    n_epochs=PHASE2_EPOCHS,
    train_loader=train_loader_256,
)

p2 = AutoencoderModule(
    model,
    optimizer,
    scheduler,
    lpips_weight=1.0,
    compute_rfid=True,
    train_only=TRAIN_ONLY_P2,
)

trainer = Trainer(
    max_epochs=PHASE2_EPOCHS,
    precision="bf16-mixed",
    gradient_clip_algorithm="norm",
    gradient_clip_val=1.0,
    deterministic=True,
)
trainer.fit(p2, train_dataloaders=train_loader_256, val_dataloaders=val_loader)
trainer.test(p2, dataloaders=val_loader)

## Phase 3 — L1 + LPIPS + PatchGAN @ 64×64 (output head only)

Freeze everything except the `out_from_channels` output conv and refine it adversarially against a PatchGAN discriminator (hinge loss + DiffAugment) on top of L1 + LPIPS. This is a manual-optimization GAN; validation stays GAN-free at 256×256.

In [ ]:
PHASE3_EPOCHS = 15
TRAIN_ONLY_P3 = ["out_from_channels"]

discriminator = PatchGANDiscriminator(in_channels=3, base_channels=64, n_layers=3)

g_params = AdversarialAutoencoderModule.trainable_params(model, TRAIN_ONLY_P3)
opt_g = torch.optim.AdamW(g_params, lr=2e-4, betas=(0.5, 0.9))
opt_d = torch.optim.AdamW(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.9))
sched_g = LinearWarmupCosineAnnealingLR(
    opt_g, warmup_steps=100, n_epochs=PHASE3_EPOCHS, train_loader=train_loader_64
)
sched_d = LinearWarmupCosineAnnealingLR(
    opt_d, warmup_steps=100, n_epochs=PHASE3_EPOCHS, train_loader=train_loader_64
)

p3 = AdversarialAutoencoderModule(
    model,
    discriminator,
    opt_g,
    opt_d,
    sched_g=sched_g,
    sched_d=sched_d,
    lpips_weight=1.0,
    gan_weight=0.1,
    compute_rfid=True,
    train_only=TRAIN_ONLY_P3,
)

trainer = Trainer(
    max_epochs=PHASE3_EPOCHS,
    precision="bf16-mixed",
    deterministic=True,
)
trainer.fit(p3, train_dataloaders=train_loader_64, val_dataloaders=val_loader)
trainer.test(p3, dataloaders=val_loader)

## Analysis

Compare 256×256 validation images against their reconstructions from the fully-trained model.

In [ ]:
# Lightning leaves the model on CPU after fit/test; move it back for inference.
model.to(DEVICE).float().eval()

images, _ = next(iter(val_loader))
with torch.no_grad():
    recons = model(images.to(DEVICE)).float().cpu()
    diffs = (images - recons).abs()

n = 8
# Row 0: originals, row 1: reconstructions, row 2: differences.
batch = torch.cat([images[:n], recons[:n], diffs[:n]]).clamp(0, 1)
grid = make_grid(batch, nrow=n)
plt.figure(figsize=(20, 6))
plt.imshow(to_pil_image(grid))
plt.title("Original (top) vs Reconstruction (bottom) @ 256x256")
plt.axis("off")
plt.show()